# AssemblyAI API Checks
Quick reference notebook for common AssemblyAI REST API operations.

**Prerequisites:** `.env` file with `ASSEMBLYAI_API_KEY` set.

### Setup: imports, config, API key

In [3]:
import os
import requests
import time
from dotenv import load_dotenv
from pprint import pprint

load_dotenv()

API_KEY = os.getenv("ASSEMBLYAI_API_KEY", "")
BASE_URL = "https://api.assemblyai.com/v2"

HEADERS = {
    "Authorization": API_KEY,
    "Content-Type": "application/json",
}

# early-exit guard: if the key is missing, stop immediately with msg
assert API_KEY, "ASSEMBLYAI_API_KEY not set — check your .env file"
print(f"API key loaded: {API_KEY[:3]}…")

API key loaded: eeb…


---
## 1. Submit a Transcription Job
Point `AUDIO_URL` at any public audio/video URL (or an AssemblyAI upload URL).

In [4]:
# AssemblyAI's audio fetcher doesn't follow redirects, so use the direct GCS URL
AUDIO_URL = "https://storage.googleapis.com/aai-docs-samples/sports_injuries.mp3"
AUDIO_URL_LOCAL = "https://cdn.assemblyai.com/upload/a3fe5806870238d7f4327976c0b9fb9720ad8745b359a632c1b06bc5f5732495/c861f045-5d68-465b-a223-c5d7e98267c1"

payload = {
    "audio_url": AUDIO_URL_LOCAL,
    "speech_models": [
        "universal-2"
    ],  # fast, cost-effective, good for general use cases. See docs for other models: https://www.assemblyai.com/docs/getting-started/models
    "language_detection": True,  # If know the language, set language_code instead (e.g. "en")
    "punctuate": True,
    "format_text": True,
}

resp = requests.post(f"{BASE_URL}/transcript", json=payload, headers=HEADERS)
resp.raise_for_status()

job = resp.json()
TRANSCRIPT_ID = job["id"]

print(f"Job submitted — ID: {TRANSCRIPT_ID}, status: {job['status']}")

Job submitted — ID: 38bf861d-4a21-4bba-87dc-56754421dddd, status: queued


---
## 2. Check Job Status by Transcript ID

In [7]:
resp = requests.get(f"{BASE_URL}/transcript/{TRANSCRIPT_ID}", headers=HEADERS)
resp.raise_for_status()

data = resp.json()
status = data["status"]

STATUS_LABEL = {
    "queued": "QUEUED",
    "processing": "PROCESSING",
    "completed": "COMPLETED",
    "error": "ERROR",
}

print(f"Status: {STATUS_LABEL.get(status, 'UNKNOWN')}")

if status == "error":
    print(f"Error: {data.get('error')}")
elif status == "completed":
    print(
        f"Words: {len(data.get('words', []))}, Duration: {data.get('audio_duration', 0):.1f}s"
    )

Status: COMPLETED
Words: 54, Duration: 28.0s


---
## 3. Poll Until Job Completes

In [6]:
POLL_INTERVAL = 5  # How many seconds to wait between each status check.

print(f"Polling {TRANSCRIPT_ID}…")

while True:
    resp = requests.get(f"{BASE_URL}/transcript/{TRANSCRIPT_ID}", headers=HEADERS)
    resp.raise_for_status()
    data = resp.json()
    status = data["status"]

    print(f"  [{time.strftime('%H:%M:%S')}] {status}")

    if status == "completed":
        print("\nDone!")
        print(data["text"][:500], "…" if len(data["text"]) > 500 else "")
        break
    elif status == "error":
        print(f"\nError: {data.get('error')}")
        break
    # else: still queued/processing — sleep and loop again

    # This prevents hammering the API unnecessarily.
    time.sleep(POLL_INTERVAL)

Polling 38bf861d-4a21-4bba-87dc-56754421dddd…
  [16:36:39] completed

Done!
Today we are going to talk about opal, which is a hydrated, amorphous form of silica and classified as a mineral due to its lack of crystalline structure. It has a lot of diffraction and influence of light through a 3D lattice of microscopic silica spheres, typically about 150 to 300 nanometers in diameter. 


---
## 4. Get Full Transcript Text by ID

In [ ]:
resp = requests.get(f"{BASE_URL}/transcript/{TRANSCRIPT_ID}", headers=HEADERS)
resp.raise_for_status()
data = resp.json()
transcript_text = data["text"]

if data["status"] != "completed":
    print(f"Not done yet — status: {data['status']}")
else:
    print("=" * 60)
    print(data["text"])
    print("=" * 60)
    print(f"\nLanguage  : {data.get('language_code', 'N/A')}")
    print(f"Duration  : {data.get('audio_duration', 0):.1f}s")
    print(f"Confidence: {data.get('confidence', 0):.2%}")

Today we are going to talk about opal, which is a hydrated, amorphous form of silica and classified as a mineral due to its lack of crystalline structure. It has a lot of diffraction and influence of light through a 3D lattice of microscopic silica spheres, typically about 150 to 300 nanometers in diameter.

Language  : en
Duration  : 28.0s
Confidence: 97.69%


---
## 5. Get Word-Level Timestamps

In [11]:
resp = requests.get(f"{BASE_URL}/transcript/{TRANSCRIPT_ID}", headers=HEADERS)
resp.raise_for_status()

# .get("words", []) — if words key is absent (e.g. job not done)
# we fall back to an empty list instead of crashing.
words = resp.json().get("words", [])

print(f"{'Word':<20} {'Start (ms)':>10} {'End (ms)':>10} {'Confidence':>12}")
print("-" * 56)

# words[:20] slices the first 20 items to avoid flooding the output.
for w in words[:20]:
    print(f"{w['text']:<20} {w['start']:>10} {w['end']:>10} {w['confidence']:>12.2%}")

if len(words) > 20:
    print(f"  … {len(words) - 20} more words")

Word                 Start (ms)   End (ms)   Confidence
--------------------------------------------------------
Today                      1280       1440       99.80%
we                         1440       1640       99.90%
are                        1640       1800       65.53%
going                      1800       1960       99.95%
to                         1960       2120       99.90%
talk                       2120       2280       99.95%
about                      2280       2520       99.90%
opal,                      2520       3080       99.46%
which                      3080       3320       99.90%
is                         3320       3600       99.85%
a                          3600       3960       99.02%
hydrated,                  3960       4760       99.50%
amorphous                  4760       5520       89.09%
form                       5680       6000       99.90%
of                         6000       6240       99.90%
silica                     6240       6960     

---
## 6. List Recent Transcripts

In [12]:
LIMIT = 10

# requests will URL-encode them automatically: ?limit=10
resp = requests.get(
    f"{BASE_URL}/transcript",
    headers=HEADERS,
    params={"limit": LIMIT},
)
resp.raise_for_status()

results = resp.json().get("transcripts", [])

print(f"{'ID':<38} {'Status':<12} {'Created':>10}")
print("-" * 62)
for t in results:
    created = t.get("created", "N/A")[:10]
    print(f"{t['id']:<38} {t['status']:<12} {created:>10}")

ID                                     Status          Created
--------------------------------------------------------------
c693a7cc-11cc-455a-a39e-9edbc987ebfd   completed    2026-03-09
7584c854-b502-4dc5-9275-07d1283c0c25   completed    2026-03-08
d7b9424f-95fa-4060-b4ea-3e16b472d7b1   completed    2026-03-08
ccef3c18-3a65-42c6-b066-d54134efe793   completed    2026-03-08
b8092ce5-3ab8-4c88-abde-daa9e436a61f   completed    2026-03-08
9d06c8ce-d63b-4e84-b9ad-6c64a1f715ed   completed    2026-03-08


---
## 7. Delete a Transcript

In [14]:
ID_TO_DELETE = "b8092ce5-3ab8-4c88-abde-daa9e436a61f"  # paste a transcript ID here

assert ID_TO_DELETE, "Set ID_TO_DELETE before running"

resp = requests.delete(f"{BASE_URL}/transcript/{ID_TO_DELETE}", headers=HEADERS)
resp.raise_for_status()

print(f"Deleted {ID_TO_DELETE}")
pprint(resp.json())

Deleted b8092ce5-3ab8-4c88-abde-daa9e436a61f
{'acoustic_model': 'assemblyai_default',
 'audio_channels': None,
 'audio_duration': 201,
 'audio_end_at': None,
 'audio_start_from': None,
 'audio_url': 'http://deleted_by_user',
 'auto_chapters': False,
 'auto_highlights': False,
 'auto_highlights_result': None,
 'boost_param': None,
 'chapters': None,
 'cluster_id': None,
 'confidence': None,
 'content_safety': None,
 'content_safety_labels': None,
 'custom_spelling': None,
 'custom_topics': False,
 'custom_topics_results': None,
 'disfluencies': False,
 'domain': None,
 'domain_options': None,
 'dual_channel': None,
 'entities': None,
 'entity_detection': False,
 'error': None,
 'filter_profanity': False,
 'format_text': False,
 'iab_categories': None,
 'iab_categories_result': None,
 'id': 'b8092ce5-3ab8-4c88-abde-daa9e436a61f',
 'is_deleted': True,
 'keyterms_prompt': [],
 'language_code': None,
 'language_codes': None,
 'language_confidence': None,
 'language_confidence_threshold': No

---
## 8. Upload a Local Audio File

In [ ]:
LOCAL_FILE = "/recorded_voice_local.mp3"

upload_headers = {"Authorization": API_KEY}

# try in read-binary mode.
try:
    with open(LOCAL_FILE, "rb") as f:
        resp = requests.post(
            "https://api.assemblyai.com/v2/upload",
            headers=upload_headers,
            data=f,
        )
    resp.raise_for_status()

    upload_url = resp.json()["upload_url"]
    print(f"Uploaded: {upload_url}")
    print("Use this as audio_url in Cell 2")
except FileNotFoundError:
    print(f"File not found: {LOCAL_FILE}")

Uploaded: https://cdn.assemblyai.com/upload/a3fe5806870238d7f4327976c0b9fb9720ad8745b359a632c1b06bc5f5732495/c861f045-5d68-465b-a223-c5d7e98267c1
Use this as audio_url in Cell 2


---
## 9. LeMUR — Ask a Question About a Transcript

In [ ]:
# LeMUR requires a paid plan, got a 401 on the free tier
# Learn: https://www.assemblyai.com/blog/lemur
# https://www.assemblyai.com/docs/llm-gateway/apply-llms-to-audio-files

QUESTION = "What is the main take-away from the transcript?"

resp = requests.post(
    "https://llm-gateway.assemblyai.com/v1/chat/completions",
    headers=HEADERS,
    json={
        "transcript_ids": [TRANSCRIPT_ID],
        "prompt": QUESTION,
        "final_model": "anthropic/claude-3-5-sonnet",  # "anthropic/claude-3-haiku" is faster/cheaper but less detailed.
        "max_output_size": 500,
    },
)

resp.raise_for_status()
print(resp.json()["response"])

HTTPError: 401 Client Error: Unauthorized for url: https://llm-gateway.assemblyai.com/v1/chat/completions

In [13]:
# {"error":"Your account does not have access to LeMUR. Please upgrade or contact us at support@assemblyai.com for more information.","status":"error","request_id":"3f6bee94-c655-4d70-943d-c110b3b6f669"}

---
## 10. Raw Response Inspector
Dump the full JSON for any transcript ID — useful for debugging.

In [14]:
# To discover all available fields and their values for a transcript
INSPECT_ID = TRANSCRIPT_ID

resp = requests.get(f"{BASE_URL}/transcript/{INSPECT_ID}", headers=HEADERS)
resp.raise_for_status()

# making it much easier to read than a raw print()
pprint(resp.json())

{'acoustic_model': 'assemblyai_default',
 'audio_duration': 28,
 'audio_end_at': None,
 'audio_start_from': None,
 'audio_url': 'https://cdn.assemblyai.com/upload/a3fe5806870238d7f4327976c0b9fb9720ad8745b359a632c1b06bc5f5732495/c861f045-5d68-465b-a223-c5d7e98267c1',
 'auto_chapters': False,
 'auto_highlights': False,
 'auto_highlights_result': None,
 'boost_param': None,
 'chapters': None,
 'confidence': 0.9769014,
 'content_safety': False,
 'content_safety_labels': {'results': [],
                           'status': 'unavailable',
                           'summary': {}},
 'custom_spelling': None,
 'custom_topics': False,
 'custom_topics_results': None,
 'disfluencies': False,
 'entities': None,
 'entity_detection': False,
 'filter_profanity': False,
 'format_text': True,
 'iab_categories': False,
 'iab_categories_result': {'results': [],
                           'status': 'unavailable',
                           'summary': {}},
 'id': 'c693a7cc-11cc-455a-a39e-9edbc987ebfd',
 'is